# Train LSTM on Colab

This notebook sets up Colab, copies a selected `data/lstm_processed/<dataset_version>` dataset to local disk, and launches a configurable source training module.

## 1. Mount Drive

In [1]:
%cd /content
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Paths

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

# Code should live on local Colab disk. Drive is only for large data/checkpoints/logs.
# Set this to your GitHub repo URL before running the cell.
GITHUB_REPO_URL = "https://github.com/sungjelly/Seoul_bike_project.git"
GITHUB_BRANCH = "main"
CODE_DIR = Path("/content/Seoul_bike_project")
RELOAD_LOCAL_CODE = True

DRIVE_ROOT = Path("/content/drive/MyDrive/Seoul_bike_project")

# Change these values to switch experiments.
DATASET_VERSION = "tts_lstm"       # examples: lstm_v1, lstm_v2
MODEL_VERSION = "tts_lstm"         # controls config/checkpoint/model/log paths
BASE_DATASET_NAME = "base"
TRAIN_MODULE = "src.training.lstm_training.train_lstm"
CONFIG_PATH = f"configs/models/lstm/{MODEL_VERSION}.yaml"
BATCH_SIZE = 32768

if CODE_DIR.exists() and RELOAD_LOCAL_CODE:
    shutil.rmtree(CODE_DIR)

if not CODE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(CODE_DIR)],
        check=True,
    )

PROJECT_DIR = CODE_DIR
DRIVE_PROCESSED_ROOT = DRIVE_ROOT / "data" / "lstm_processed"
LOCAL_PROCESSED_ROOT = Path("/content/lstm_processed")
DRIVE_DATA_DIR = DRIVE_PROCESSED_ROOT / DATASET_VERSION
DRIVE_BASE_DATA_DIR = DRIVE_PROCESSED_ROOT / BASE_DATASET_NAME
LOCAL_DATA_DIR = LOCAL_PROCESSED_ROOT / DATASET_VERSION
LOCAL_BASE_DATA_DIR = LOCAL_PROCESSED_ROOT / BASE_DATASET_NAME
DATA_DIR = LOCAL_DATA_DIR

CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / MODEL_VERSION
MODEL_DIR = DRIVE_ROOT / "models" / MODEL_VERSION
LOG_DIR = DRIVE_ROOT / "logs" / MODEL_VERSION

os.chdir(PROJECT_DIR)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Code directory:", PROJECT_DIR)
print("Drive artifact root:", DRIVE_ROOT)
print("Drive dataset:", DRIVE_DATA_DIR)
print("Drive base dataset:", DRIVE_BASE_DATA_DIR)
print("Local dataset:", DATA_DIR)
print("Training module:", TRAIN_MODULE)
print("Config path:", CONFIG_PATH)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Current working directory:", Path.cwd())

Code directory: /content/Seoul_bike_project
Drive artifact root: /content/drive/MyDrive/Seoul_bike_project
Drive dataset: /content/drive/MyDrive/Seoul_bike_project/data/lstm_processed/tts_lstm
Drive base dataset: /content/drive/MyDrive/Seoul_bike_project/data/lstm_processed/base
Local dataset: /content/lstm_processed/tts_lstm
Training module: src.training.lstm_training.train_lstm
Config path: configs/models/lstm/tts_lstm_v2.yaml
Checkpoint directory: /content/drive/MyDrive/Seoul_bike_project/checkpoints/tts_lstm_v2
Current working directory: /content/Seoul_bike_project


## 3. Install Requirements

In [3]:
%cd /content/Seoul_bike_project
!python -m pip install -r requirements.txt

/content


## 4. Copy Dataset to Colab Disk

In [4]:
import shutil

RELOAD_LOCAL_DATA = True
for drive_dir, local_dir in [(DRIVE_BASE_DATA_DIR, LOCAL_BASE_DATA_DIR), (DRIVE_DATA_DIR, LOCAL_DATA_DIR)]:
    if not drive_dir.exists():
        raise FileNotFoundError(f"Missing dataset directory: {drive_dir}")
    if local_dir.exists() and RELOAD_LOCAL_DATA:
        shutil.rmtree(local_dir)
    if not local_dir.exists():
        shutil.copytree(drive_dir, local_dir)
print("Using local dataset:", DATA_DIR)

Using local dataset: /content/lstm_processed/tts_lstm


## 5. Verify Dataset

In [5]:
import json

base_data_path = DATA_DIR / "base_data.json"
array_dir = DATA_DIR
if base_data_path.exists():
    with open(base_data_path, "r", encoding="utf-8") as f:
        base_data = json.load(f)
    array_dir = (DATA_DIR / str(base_data["base_data_dir"]).replace("\\", "/")).resolve()

required = [
    array_dir / "dynamic_features.npy",
    array_dir / "targets.npy",
    array_dir / "targets_raw.npy",
    array_dir / "static_numeric.npy",
    DATA_DIR / "sample_index_train.npy",
    DATA_DIR / "sample_index_val.npy",
    DATA_DIR / "feature_config.json",
    DATA_DIR / "dataset_summary.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing dataset files:\n" + "\n".join(missing))
with open(DATA_DIR / "dataset_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)
print(json.dumps({
    "T_total": summary["T_total"],
    "S": summary["S"],
    "samples_per_split": summary["samples_per_split"],
}, indent=2))

{
  "T_total": 21888,
  "S": 2780,
  "samples_per_split": {
    "train": 33626880,
    "val": 4136640,
    "test_2025_winter": 8139840,
    "test_2024_april_june": 9340800
  }
}


## 6. W&B Login

In [6]:
import os
from getpass import getpass
import wandb

wandb_key = os.environ.get("WANDB_API_KEY")

try:
    from google.colab import userdata
    wandb_key = wandb_key or userdata.get("WANDB_API_KEY")
except Exception as exc:
    print("Colab Secrets not available from this kernel:", type(exc).__name__)

if not wandb_key:
    wandb_key = getpass("Paste W&B API key: ")

os.environ["WANDB_API_KEY"] = wandb_key.strip()
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)

Colab Secrets not available from this kernel: TimeoutException


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sungjelly (sungjelly-kaist-digital-humanities-and-social-sciences-g) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 7. Train With Source Script

In [7]:
!python -m {TRAIN_MODULE} \
  --config {CONFIG_PATH} \
  --data_dir {DATA_DIR} \
  --checkpoint_dir {CHECKPOINT_DIR} \
  --model_dir {MODEL_DIR} \
  --log_dir {LOG_DIR} \
  # --batch_size {BATCH_SIZE} \
  --resume auto \
  --resume_mode auto \
  --wandb_enabled true \
  --smoke_test true

Smoke test passed. loss=0.431727
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sungjelly (sungjelly-kaist-digital-humanities-and-social-sciences-g) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run ob9bcjnq (0.5s)
wandb: ⣯ setting up run ob9bcjnq (0.5s)
wandb: ⣟ setting up run ob9bcjnq (0.5s)
wandb: ⡿ setting up run ob9bcjnq (0.5s)
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/Seoul_bike_project/wandb/run-20260505_122419-ob9bcjnq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tts_lstm_v2
wandb: ⭐️ View project at https://wandb.ai/sungjelly-kaist-digital-humanities-and-social-sciences-g/Seoul_Bike_Project
wandb: 🚀 View run at https://wandb.ai/sungjelly-kaist-digit